# 07a - Fitting K-Means and Choosing Groups

Last time, K-means divided 72 simulated customers using two measurements: requests last term and average minutes per request. We compared two- and three-group results. Today we will look more closely at how K-means fits those groups and what evidence might help us choose among them.

The simulation has no built-in customer types. A useful question is whether a proposed grouping would help a service team investigate or plan support. A fitted partition alone cannot answer that question.

## References

- [Scikit-learn K-means guide](https://scikit-learn.org/stable/modules/clustering.html#k-means): the squared-distance objective, algorithm, initialization, and limits.
- [Scikit-learn `KMeans` reference](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html): fitting options and the values stored after a fit.
- [Scikit-learn silhouette guide](https://scikit-learn.org/stable/modules/clustering.html#silhouette-coefficient): a measure of within-group closeness and between-group separation.

## How K-Means Forms Groups

K-means starts with a requested number of groups, $K$. It assigns each point to the nearest current center, then moves each center to the mean of its assigned points. It repeats those steps until the centers settle or an iteration limit is reached. Different starting centers can produce different results, so a fit may try several starts.

```text
For each start:
    Choose K initial centers.
    Repeat:
        Assign every point to its nearest center.
        Replace each center with the mean of its assigned points.
    Stop when the centers move little enough or an iteration limit is reached.
    Compute the inertia of that start.
Keep the completed run with the lowest final inertia.
```

Next we will define inertia, then calculate it in a four-point example.

## Inertia, the Total Squared Distance

K-means tries to put each observation near the center of its assigned group. For each observation, square its distance to that center, then add the squared distances over all observations. Scikit-learn calls this total **inertia**. A smaller value means a tighter fit in the chosen feature space.

If $z_i$ is observation $i$ in the feature space given to K-means, and $\mu_{g(i)}$ is the center of its assigned group, the total is

$$
\text{inertia} = \sum_i \left\lVert z_i - \mu_{g(i)} \right\rVert^2
$$

Assignment chooses the smallest squared distance for each point when the centers are fixed. The mean minimizes the squared-distance total for a group's points when its assignments are fixed. Each step reduces or preserves inertia under that condition.

The next example has four points on a number line. We ask for two groups and start with centers at 1 and 2. Inspect the points before following the fit.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

points = np.array([1.0, 2.0, 8.0, 9.0])
centers = np.array([1.0, 2.0])
display(pd.DataFrame({"Point": points.astype(int)}).T)
print(f"Starting centers: {centers.astype(int).tolist()}")

Both centers start on the left. Assign each point to its nearer center. What will points 8 and 9 contribute to the total squared distance? The next cell repeats assignment and center updates three times.

In [ ]:
history = []

for round_number in range(3):
    distances = np.abs(points[:, None] - centers)
    groups = distances.argmin(axis=1)
    history.append(
        {
            "Round": round_number,
            "Centers": np.round(centers, 2).tolist(),
            "Group 0": points[groups == 0].astype(int).tolist(),
            "Group 1": points[groups == 1].astype(int).tolist(),
            "Total squared distance": ((points - centers[groups]) ** 2).sum(),
        }
    )
    centers = np.array([points[groups == group].mean() for group in (0, 1)])

display(pd.DataFrame(history).round(2))

At the start, points 8 and 9 contribute $(8-2)^2=36$ and $(9-2)^2=49$. Points 1 and 2 contribute zero, so the total is 85. The new center for points 2, 8, and 9 is $(2+8+9)/3=6.33$. On the next assignment, point 2 moves to the left group. The final centers are 1.5 and 8.5, and the total falls to 1.

The figure shows the starting and final positions. The arrows at the top span the distances that contribute 36 and 49 to the starting total. This view explains the squared-distance calculation; the center-movement sequence from 06b showed the updates in two dimensions.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 3.6), sharex=True, layout="constrained")
for ax, y, current_centers, title in (
    (axes[0], 0, [1, 2], "Start: total squared distance = 85"),
    (axes[1], 0, [1.5, 8.5], "Settled: total squared distance = 1"),
):
    ax.scatter(points, np.full_like(points, y), s=80, color="#4477AA", label="Point")
    ax.scatter(current_centers, [y - 0.2, y - 0.2], marker="X", s=180,
               color="#EE6677", label="Center")
    ax.set(ylim=(-0.6, 0.7), yticks=[], title=title)
    ax.grid(axis="x", alpha=0.2)
for point, label, height in ((8, r"$6^2=36$", 0.25), (9, r"$7^2=49$", 0.48)):
    axes[0].annotate("", xy=(point, height), xytext=(2, height),
                     arrowprops={"arrowstyle": "<->", "color": "0.35"})
    axes[0].text((2 + point) / 2, height + 0.04, label, ha="center")
axes[1].set(xlabel="Point on the number line", xlim=(0, 10), xticks=range(1, 10))
axes[0].legend(loc="lower center", ncol=2)
plt.show()

## From the Algorithm to Scikit-Learn

Scikit-learn uses a recurring sequence:

1. **Instantiate:** Create an object and set its options, such as `StandardScaler()` or `KMeans(n_clusters=2)`.
2. **Fit:** Give the object data. `StandardScaler.fit()` learns column means and standard deviations; `KMeans.fit()` learns centers and assignments.
3. **Inspect:** Read fitted results from attributes ending in `_` by convention, such as `mean_`, `cluster_centers_`, `labels_`, and `inertia_`. Constructor settings such as `n_clusters` have no trailing underscore.
4. **Apply:** Use the fitted scaler's `transform()` to scale rows, then the fitted K-means model's `predict()` to assign a group. `fit_transform()` combines fitting and transforming the initial data. `KMeans.transform()` instead returns distances to centers.

We will use separate fit and transform calls below so each step is visible.

## Return to the Customers

We will use the same customer generator as last time. Each row is one customer; the two columns record request count and average minutes per request. Inspect the first five rows before fitting a model.

In [ ]:
rng = np.random.default_rng(7130)
customers = pd.DataFrame(
    {
        "requests": rng.integers(1, 13, size=72),
        "avg_minutes": rng.integers(8, 65, size=72),
    }
)
customers.index = pd.Index(range(1, 73), name="customer")

display(customers.head())

Minutes span a wider numerical range than request counts. We fit a scaler to the 72 rows, transform them, and then fit K-means to the standardized coordinates. The scaler and K-means are separate fitted objects, each with quantities we will use again.

In [ ]:
scaler = StandardScaler()
scaler.fit(customers)
scaled_customers = scaler.transform(customers)
two_groups = KMeans(n_clusters=2, init="k-means++", n_init=20, random_state=7130)
two_groups.fit(scaled_customers)
print(f"Scaled data shape: {scaled_customers.shape}")
print(f"Fitted centers shape: {two_groups.cluster_centers_.shape}")

The scaler has learned two means and two standard deviations. K-means has learned two centers in that standardized space and one group assignment for each customer. The shapes tell us that 72 customer rows with two features produced two centers with two coordinates each.

## Which Center Is Closer?

**The model uses standardized distances.** The solid line is its decision boundary: points on either side are assigned to different centers. Both panels show the same fit. The left panel displays it in original units; the right panel displays it in standardized coordinates. Customer 22 is highlighted in both.

The dashed line in the left panel shows a different rule: where distances to the same centers would tie if we measured them in original units. It is a comparison, not another fitted model. Does customer 22 fall on the same side of both lines?

In [ ]:
from matplotlib.colors import ListedColormap

raw_x = np.linspace(0.5, 12.5, 500)
raw_y = np.linspace(5, 67, 500)
raw_grid_x, raw_grid_y = np.meshgrid(raw_x, raw_y)
raw_grid = pd.DataFrame({"requests": raw_grid_x.ravel(), "avg_minutes": raw_grid_y.ravel()})
raw_regions = two_groups.predict(scaler.transform(raw_grid)).reshape(raw_grid_x.shape)

scaled_x = np.linspace(scaled_customers[:, 0].min() - 0.2, scaled_customers[:, 0].max() + 0.2, 500)
scaled_y = np.linspace(scaled_customers[:, 1].min() - 0.2, scaled_customers[:, 1].max() + 0.2, 500)
scaled_grid_x, scaled_grid_y = np.meshgrid(scaled_x, scaled_y)
scaled_grid = np.column_stack([scaled_grid_x.ravel(), scaled_grid_y.ravel()])
scaled_regions = two_groups.predict(scaled_grid).reshape(scaled_grid_x.shape)

center_raw = scaler.inverse_transform(two_groups.cluster_centers_)
customer_22 = customers.loc[[22]]
customer_22_scaled = scaler.transform(customer_22)[0]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3), layout="constrained")
panels = (
    (axes[0], raw_grid_x, raw_grid_y, raw_regions, customers.to_numpy(), center_raw,
     customer_22.to_numpy()[0], "Original units", "Requests last term", "Average minutes"),
    (axes[1], scaled_grid_x, scaled_grid_y, scaled_regions, scaled_customers,
     two_groups.cluster_centers_, customer_22_scaled, "Standardized coordinates",
     "Standardized requests", "Standardized minutes"),
)
for (ax, grid_x, grid_y, regions, rows,
     panel_centers, customer_point, title, xlabel, ylabel) in panels:
    ax.contourf(grid_x, grid_y, regions, levels=[-0.5, 0.5, 1.5],
                cmap=ListedColormap(["#DDEAF5", "#F8E4E7"]), alpha=0.8)
    ax.contour(grid_x, grid_y, regions, levels=[0.5], colors="0.3", linewidths=1)
    ax.scatter(rows[:, 0], rows[:, 1], s=18, color="0.45", alpha=0.7)
    ax.scatter(panel_centers[:, 0], panel_centers[:, 1], marker="X", s=150,
               color="black", label="Fitted centers")
    ax.scatter(*customer_point, marker="*", s=210, color="#FFCC33",
               edgecolor="black", linewidth=0.8, label="Customer 22")
    for group, center in enumerate(panel_centers):
        ax.annotate(f"{group}", center, xytext=(7, 7), textcoords="offset points")
    ax.set(xlabel=xlabel, ylabel=ylabel, title=title)
    ax.grid(alpha=0.15)
raw_center_gap = center_raw[1] - center_raw[0]
raw_midpoint = center_raw.mean(axis=0)
raw_distance_boundary_x = (
    raw_midpoint[0]
    - (raw_center_gap[1] / raw_center_gap[0]) * (raw_y - raw_midpoint[1])
)
axes[0].plot(raw_distance_boundary_x, raw_y, color="#AA3377", linestyle="--",
             linewidth=2, label="Boundary using raw distances")
axes[0].plot([], [], color="0.3", linewidth=1, label="Fitted boundary")
axes[0].set(xlim=(raw_x.min(), raw_x.max()), ylim=(raw_y.min(), raw_y.max()))
axes[1].set_aspect("equal")
axes[0].legend(loc="upper left", fontsize=8)
fig.suptitle("Fitted boundary and a raw-distance comparison")
plt.show()

**Scaling changes which center is closest to customer 22.** The solid boundary puts this customer in group 1. The dashed comparison would put it in group 0. The table checks the distances behind those assignments.

In [ ]:
centers_scaled = two_groups.cluster_centers_
centers_original = scaler.inverse_transform(centers_scaled)

distance_table = pd.DataFrame(
    {
        "Group": [0, 1],
        "Distance after scaling": np.linalg.norm(centers_scaled - customer_22_scaled, axis=1),
        "Distance in original units": np.linalg.norm(
            centers_original - customer_22.to_numpy()[0], axis=1
        ),
    }
)
display(distance_table.round(2))
print(f"Customer 22 is assigned to group {two_groups.labels_[21]}.")

Customer 22 has six requests and an average of 44 minutes per request. After scaling, its distance to group 1's center is 1.05, compared with 1.13 to group 0's center. In original units, group 0's center is closer: 9.56 versus 10.94. The fitted model uses the scaled comparison, so it assigns customer 22 to group 1.

**The feature scale matters.** Standardizing prevents minutes from dominating solely because they have larger numerical values. It does not establish that requests and minutes matter equally to the service team. Feature choice and weighting remain analytical decisions.

## Starting and Stopping a Fit

Each K-means run needs starting centers. The default `init="k-means++"` favors observations far from centers already chosen, which generally gives the fit a better start than selecting observations uniformly at random. It changes how a run starts, not the assignment-and-update steps that follow. In our four-point example, if 1 were chosen first, k-means++ would favor a distant point such as 8 or 9 next. `init="random"` instead samples observed rows without that distance preference. We can also supply initial centers or a function that chooses them.

The `n_init` setting says how many complete runs to try from different starts; K-means keeps the run with the lowest final inertia. In the course's scikit-learn version, `n_init="auto"` means one run with k-means++, ten with random or a function, and one with supplied centers. We set `n_init=20` to reduce dependence on a single start in this example and to compare one start with the best of 20 below. Twenty is a teaching choice, not a required value.

These are constructor settings: we choose them when creating `KMeans(...)`, and `fit(scaled_customers)` uses them with the supplied data. The last column shows defaults in the course's scikit-learn version. `max_iter` and `tol` control stopping within each run; we leave them at their defaults.

| Constructor setting | What it controls | Default |
| --- | --- | --- |
| `n_clusters=2` | How many groups to form. | `8` |
| `init="k-means++"` | How each run chooses initial centers. | `"k-means++"` |
| `n_init=20` | How many complete runs to try; retain the lowest final inertia. | `"auto"` |
| `random_state=7130` | Makes the random choices repeatable. | `None` |
| `max_iter` (omitted) | Caps the assignment-and-update rounds in each run. | `300` |
| `tol` (omitted) | Stops a run when centers move little enough. | `0.0001` |

The initial centers chosen by k-means++ or random are observed data points; later mean updates can put centers between observed points. Scikit-learn also offers `algorithm` as a computational choice. Its default is `"lloyd"`; changing it does not change the squared-distance objective. The next comparison keeps $K$, initialization method, data, and random seed the same while changing only the number of starts.

In [ ]:
single_start = KMeans(n_clusters=2, init="k-means++", n_init=1, random_state=7).fit(
    scaled_customers
)
twenty_starts = KMeans(n_clusters=2, init="k-means++", n_init=20, random_state=7).fit(
    scaled_customers
)
print(f"One start inertia: {single_start.inertia_:.2f}")
print(f"Twenty starts, best inertia: {twenty_starts.inertia_:.2f}")

These scores summarize different partitions of the same customers. The figure shows which split each fit retained. Compare the group boundaries suggested by the point colors and centers. Group numbers and colors are local to each panel.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4), sharex=True, sharey=True, layout="constrained")
for ax, model, title in zip(
    axes, (single_start, twenty_starts), ("One start", "Best of 20 starts"), strict=True
):
    for group, color in ((0, "#4477AA"), (1, "#EE6677")):
        members = customers.iloc[np.flatnonzero(model.labels_ == group)]
        ax.scatter(members["requests"], members["avg_minutes"], color=color,
                   s=35, alpha=0.8, label=f"Group {group}")
    fitted_centers = scaler.inverse_transform(model.cluster_centers_)
    ax.scatter(fitted_centers[:, 0], fitted_centers[:, 1], marker="X", color="black", s=150)
    ax.set(xlabel="Requests last term", title=f"{title}: inertia {model.inertia_:.1f}")
    ax.grid(alpha=0.2)
axes[0].set_ylabel("Average minutes per request")
axes[0].legend()
fig.suptitle("More starts can find a tighter partition")
plt.show()

The one-start fit has inertia about 95.6; the best of 20 starts is about 90.8. For `n_init=20`, scikit-learn selects initial centers, runs the full assignment-and-update loop, and calculates final inertia 20 times. It retains the completed run with the lowest **final** inertia, not the start with the lowest initial inertia or the most frequent partition. This comparison does not tell us that the customer groups are stable or useful.

## Read the Fitted Result

The fitted model stores each customer's group in `labels_`, its centers in `cluster_centers_`, its total squared distance in `inertia_`, and the number of iterations in `n_iter_`. The stored centers are in standardized coordinates. We convert them back to requests and minutes before describing them.

In [ ]:
center_table = pd.DataFrame(
    scaler.inverse_transform(two_groups.cluster_centers_),
    columns=["Center requests", "Center minutes"],
)
center_table.index.name = "Group"

print(f"Inertia: {two_groups.inertia_:.2f}; iterations: {two_groups.n_iter_}")
display(center_table.round(1))

The centers are averages in the two-feature space, not necessarily actual customers. To describe the members of each group, we now combine the model's assignments with the original customer records.

In [ ]:
profile_table = (
    customers.assign(group=two_groups.labels_)
    .groupby("group")
    .agg(
        customers=("requests", "size"),
        mean_requests=("requests", "mean"),
        mean_minutes=("avg_minutes", "mean"),
    )
)

display(profile_table.round(1))

One group averages about 9.2 requests and the other about 3.4; their average request times are similar. The centers and profiles describe this fitted partition. Group numbers are arbitrary names, and differences in the two fitting features do not independently prove different service needs.

A later customer can be assigned without fitting new centers. We apply the **fitted scaler** to that customer's measurements, then ask the fitted K-means model for the nearest group. This assignment is a starting point for investigation, not a service recommendation.

In [ ]:
new_customer = pd.DataFrame({"requests": [10], "avg_minutes": [40]})
new_customer_scaled = scaler.transform(new_customer)
print(f"New customer assigned to group {two_groups.predict(new_customer_scaled)[0]}.")

## How Many Groups?

We chose two groups above because we asked for two. An **elbow plot** shows how inertia changes as we request more groups. More centers can fit the observed points more closely, so a low value by itself cannot choose $K$.

A **silhouette score** asks another question: are points closer to their own group than to the next nearest group? For one point, let $a$ be its average distance to other points in its group and $b$ its average distance to points in the nearest other group. Its silhouette is $(b-a)/\max(a,b)$. Scores near 1 indicate separation; scores near 0 indicate overlap; negative scores suggest the point is closer to another group on average. The mean over points summarizes one fitted partition. It is undefined for $K=1$ because there is no other group.

The next cell fits $K=1$ through $8$ on the same standardized customers with the same settings. It records inertia and mean silhouette for each fit.

In [ ]:
candidate_k = range(1, 9)
candidate_models = {
    k: KMeans(n_clusters=k, init="k-means++", n_init=20, random_state=7130).fit(
        scaled_customers
    )
    for k in candidate_k
}
inertias = [candidate_models[k].inertia_ for k in candidate_k]
silhouettes = [np.nan] + [
    silhouette_score(scaled_customers, candidate_models[k].labels_)
    for k in range(2, 9)
]
diagnostics = pd.DataFrame(
    {"K": list(candidate_k), "Inertia": inertias, "Mean silhouette": silhouettes}
)
display(diagnostics.round(3))

The two panels put the diagnostics side by side. Look for a marked bend in the first curve and a clear high score in the second.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4), layout="constrained")
axes[0].plot(diagnostics["K"], diagnostics["Inertia"], marker="o")
axes[0].set(
    xlabel="Requested number of groups (K)",
    ylabel="Total squared distance",
    title="Inertia: lower with more centers",
    xticks=[1, 2, 3, 4, 6, 8],
)
axes[1].plot(diagnostics["K"], diagnostics["Mean silhouette"], marker="o", color="#AA3377")
axes[1].set(
    xlabel="Requested number of groups (K)",
    ylabel="Mean silhouette",
    title="Silhouette: closeness and separation",
    xticks=[1, 2, 3, 4, 6, 8],
    ylim=(0, 0.6),
)
for ax in axes:
    ax.grid(alpha=0.2)
fig.suptitle("Two diagnostics for the same customer fits")
plt.show()

Inertia falls from 144 at $K=1$ to about 91 at $K=2$ and 54 at $K=3$; later improvements are smaller. The mean silhouette values from $K=3$ onward are close to one another, roughly 0.40 to 0.42. Neither curve points to a clear, known number of customer types in these simulated data. Both assess the fitted geometry, not whether a service team needs distinct responses.

### Pause: What Does the Third Group Add?

Where would you place an elbow? Last time, the three-group fit produced a group with longer requests. What could that distinction change for the service team? Do the two diagnostics establish that three groups are the right choice?

##### Answer

The large reductions through $K=3$ make three groups worth examining, but the bend is not decisive and nearby silhouette scores are similar. The longer-request group might motivate an investigation of specialized help or longer appointments. We would need real request records and another period of customer data before recommending a service change.

## Can Similar Fits Tell Different Stories?

Repeated starts protect against accepting one poor start as the only fit. They do not guarantee that one low-inertia partition is the only plausible one. The two fits below each use 20 starts but use different random states. Their decision boundaries show the areas assigned to each fitted center. Compare those areas before reading the scores.

In [ ]:
other_two_groups = KMeans(
    n_clusters=2, init="k-means++", n_init=20, random_state=8
).fit(scaled_customers)

The plotted customers are the same in both panels. Only the fitted centers and assignments differ.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True, sharey=True, layout="constrained")
for ax, model, title in zip(
    axes, (two_groups, other_two_groups), ("Fit A", "Fit B"), strict=True
):
    regions = model.predict(scaler.transform(raw_grid)).reshape(raw_grid_x.shape)
    ax.contourf(raw_grid_x, raw_grid_y, regions, levels=[-0.5, 0.5, 1.5],
                cmap=ListedColormap(["#DDEAF5", "#F8E4E7"]), alpha=0.8)
    ax.contour(raw_grid_x, raw_grid_y, regions, levels=[0.5], colors="0.3", linewidths=1)
    for group, marker, color in ((0, "o", "#4477AA"), (1, "^", "#EE6677")):
        members = customers.iloc[np.flatnonzero(model.labels_ == group)]
        ax.scatter(
            members["requests"], members["avg_minutes"],
            marker=marker, color=color, s=35, alpha=0.75, label=f"Group {group}",
        )
    fitted_centers = scaler.inverse_transform(model.cluster_centers_)
    ax.scatter(fitted_centers[:, 0], fitted_centers[:, 1], marker="X", color="black", s=150)
    ax.set(xlabel="Requests last term", title=title)
    ax.grid(alpha=0.2)
    ax.legend()
axes[0].set_ylabel("Average minutes per request")
fig.suptitle("Two low-inertia ways to divide the same customers")
plt.show()

print(f"Fit A inertia: {two_groups.inertia_:.3f}")
print(f"Fit B inertia: {other_two_groups.inertia_:.3f}")

Fit A mainly separates customers by request count. Fit B mainly separates them by minutes per request. Their inertia values differ by only about 0.12, yet their stories for the service team are quite different. Group 0 in one fit is not automatically the same group as group 0 in the other.

These records have no clear gaps that would settle the choice visually. K-means favors compact groups under squared distance; elongated patterns and unusually distant points can also pull its centers into unhelpful positions. Inspect the data geometry and the resulting assignments before attaching an operational meaning to a group.

### Pause: What Can We Conclude?

Which of these two groupings would you take to the service team for further investigation? What would you check before changing how support is provided?

##### Answer

Either fit could motivate a question: frequent customers may need a different response from customers with lengthy requests. The nearly equal scores do not settle which question matters more. We would inspect real request content, ask the team what distinctions affect its work, and see whether a similar pattern appears in another period. Comparing random starts checks sensitivity to fitting; another period checks sensitivity to which customers were observed. These simulated records do not justify a service change.

K-means gives us a partition under chosen features, scale, $K$, and fitting settings. The total squared distance tells us how tightly that partition fits those coordinates. Comparing fits and asking what the groups would change tells us what evidence to seek next.